<h1 style='color:#A23B72'>Análisis Regional: Aplicando Parámetros Distintos por Segmento</h1>

<p style='color:#b0b0b0'>Dividimos curvas en regiones y optimizamos cada una independientemente. Diferentes partes pueden tener características que requieren parámetros distintos.</p>

## 1. Motivación

Una curva puede tener heterogeneidad interna:
- Inicio suave → pocas campanas
- Centro con picos → más campanas
- Final con variación → método distinto

Dividir en K regiones y optimizar cada una independientemente puede mejorar desempeño global.

## 2. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.signal import find_peaks
from scipy.optimize import curve_fit

plt.rcParams.update({'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False})

ROOT = Path.cwd().parent
DATA_T = ROOT / 'datos' / 'target'

def leer_target(cid):
    return pd.read_csv(DATA_T / f'curve_{cid:04d}.txt', header=None, names=['x', 'y']).sort_values('x').reset_index(drop=True)

print('Setup completado')

## 3. Segmentación de Curvas

In [ ]:
def segmentar_uniforme(curva, n_regiones=3):
    x = curva['x'].values
    y = curva['y'].values
    x_min, x_max = x.min(), x.max()
    ancho = (x_max - x_min) / n_regiones
    
    regiones = []
    for i in range(n_regiones):
        inicio = x_min + i * ancho
        fin = x_min + (i + 1) * ancho
        mask = (x >= inicio) & (x <= fin)
        if mask.sum() > 0:
            regiones.append((x[mask], y[mask], f'Región {i+1}'))
    return regiones

print('Segmentación uniforme lista')

## 4. Optimización Regional

In [ ]:
def suma_gaussianas(x, *params):
    n = (len(params) - 1) // 3
    c = params[-1]
    y = np.full_like(x, c, dtype=float)
    for i in range(n):
        A, mu, sig = params[3*i], params[3*i+1], params[3*i+2]
        y += A * np.exp(-(x - mu)**2 / (2 * sig**2 + 1e-9))
    return y

def optimizar_region(x_seg, y_seg, n_campanas=2):
    n = max(1, min(n_campanas, 5))
    p0 = []
    for _ in range(n):
        p0 += [y_seg.max() - y_seg.min(), x_seg.mean(), (x_seg.max() - x_seg.min())/4]
    p0.append(y_seg.mean())
    
    try:
        popt, _ = curve_fit(suma_gaussianas, x_seg, y_seg, p0=p0, maxfev=2000)
        y_pred = suma_gaussianas(x_seg, *popt)
        ss_res = np.sum((y_seg - y_pred)**2)
        ss_tot = np.sum((y_seg - y_seg.mean())**2)
        r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
        return y_pred, r2
    except:
        return y_seg, 0.0

print('Optimización regional lista')

## 5. Ejemplo: Curva 25

In [ ]:
curva = leer_target(25)
regiones = segmentar_uniforme(curva, 3)

print('Optimización Regional - Curva 25')
print('='*50)

resultados = []
for i, (x_seg, y_seg, label) in enumerate(regiones):
    n_camp = [2, 3, 2][i]
    y_pred, r2 = optimizar_region(x_seg, y_seg, n_campanas=n_camp)
    resultados.append({'region': i+1, 'n': n_camp, 'r2': r2, 'y_pred': y_pred, 'x': x_seg, 'y': y_seg})
    print(f'Región {i+1}: n={n_camp}, R²={r2:.4f}')

r2_prom = np.mean([r['r2'] for r in resultados])
print(f'\nR² Promedio Regional: {r2_prom:.4f}')

## 6. Visualización

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor('white')

colors = ['#FF6B6B', '#4ECDC4', '#FFE66D']

# Plot target
ax.plot(curva['x'], curva['y'], color='#e0e0e0', lw=2, label='Target', zorder=1)

# Plot regiones optimizadas
for res, color in zip(resultados, colors):
    ax.plot(res['x'], res['y_pred'], color=color, lw=2, label=f"Región {res['region']} (R²={res['r2']:.4f})", zorder=2)

ax.set_title(f'Análisis Regional: Curva 25 | R² Promedio: {r2_prom:.4f}', fontweight='bold', fontsize=12)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Conclusiones

### Cuándo usar análisis regional:

- Curvas con heterogeneidad clara → mejor con regional
- R² global < 0.98 → considerar regional
- K ≤ 5 regiones → manejable

### Próximos pasos:

1. Escalar a 500 curvas
2. Comparar global vs regional
3. Combinar con notebook 02 (análisis de características)